In [14]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_WORKBOOK = Path("..") / "data" / "raw" / "kenya_maize_production_2019_2025_v2.xlsx"
PROCESSED_DIR = Path("..") / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load the data sheets

In [15]:
xls = pd.ExcelFile(RAW_WORKBOOK)

national_es = pd.read_excel(xls, sheet_name="National_ES_2021_2025")
supply_use = pd.read_excel(xls, sheet_name="Supply_use_2024_2025")
national_afa = pd.read_excel(xls, sheet_name="National_AFA_2019_2024")
county = pd.read_excel(xls, sheet_name="County_2020_2024")

raw_sheets = {
    "national_es": national_es,
    "supply_use": supply_use,
    "national_afa": national_afa,
    "county": county,
}
for name, df in raw_sheets.items():
    print(f"{name:<13} shape={df.shape}")

national_es   shape=(5, 8)
supply_use    shape=(2, 14)
national_afa  shape=(6, 7)
county        shape=(47, 11)


## 2. Data quality checks

In [16]:
for name, df in raw_sheets.items():
    numeric = df.select_dtypes("number")
    print(
        f"{name:<13} missing={df.isna().sum().sum():>3}  "
        f"duplicate rows={df.duplicated().sum():>2}  "
        f"negative={(numeric < 0).sum().sum():>2}  "
        f"zero={(numeric == 0).sum().sum():>2}"
    )

national_es   missing=  0  duplicate rows= 0  negative= 0  zero= 0
supply_use    missing=  0  duplicate rows= 0  negative= 0  zero= 1
national_afa  missing=  0  duplicate rows= 0  negative= 0  zero= 0
county        missing=  0  duplicate rows= 0  negative= 0  zero= 1


In [17]:

for name, df in raw_sheets.items():
    for col in df.select_dtypes("number").columns:
        zero_rows = df.index[df[col] == 0].tolist()
        if zero_rows:
            years = df.loc[zero_rows, "Year"].tolist() if "Year" in df else zero_rows
            print(f"{name} | {col} | year/row: {years}")

supply_use | Feed ('000 t) | year/row: [2025]
county | Production (t) 2021 | year/row: [27]


In [18]:
# County-specific checks
print("Unique counties:", county["County"].nunique(), "of", len(county), "rows")
print("Names with leading/trailing spaces:", (county["County"] != county["County"].str.strip()).sum())

# Each year should appear once in the national sheets
for name, df in raw_sheets.items():
    if "Year" in df:
        print(f"{name:<13} duplicated years: {df['Year'].duplicated().sum()}")

Unique counties: 47 of 47 rows
Names with leading/trailing spaces: 0
national_es   duplicated years: 0
supply_use    duplicated years: 0
national_afa  duplicated years: 0


## 3. Reshape the county data to long format

In [19]:
# stack all year columns into two columns, "column" (old name) and "value"
county_long = county.melt(id_vars="County", var_name="column", value_name="value")

# split names such as "Area (ha) 2020" into a measure and a year
parts = county_long["column"].str.extract(r"^(?P<measure>.+) (?P<year>\d{4})$")
assert parts.notna().all().all(), "Some column names did not match the expected pattern"
county_long = county_long.join(parts)
county_long["year"] = county_long["year"].astype(int)

# put area and production side by side again, one row per county-year
county_long = (
    county_long.pivot(index=["County", "year"], columns="measure", values="value")
    .reset_index()
    .rename(columns={"County": "county", "Area (ha)": "area_ha", "Production (t)": "production_t"})
)
county_long.columns.name = None

county_long.head()

,county,year,area_ha,production_t
0,Baringo,2020,47437,68374
1,Baringo,2021,34709,59169
2,Baringo,2022,37894,50820
3,Baringo,2023,42501,94877
4,Baringo,2024,47457,83958


## 4. Add yield and flag very small maize areas

In [20]:
county_long["yield_t_ha"] = county_long["production_t"] / county_long["area_ha"]
county_long["yield_bags_ha"] = county_long["yield_t_ha"] * 1000 / 90

MIN_AVG_AREA_HA = 1000  

avg_area = county_long.groupby("county")["area_ha"].mean()
small_counties = avg_area[avg_area < MIN_AVG_AREA_HA].sort_values()
county_long["small_area_flag"] = county_long["county"].isin(small_counties.index)

print(small_counties.round(0).to_string())

county
Wajir      127.0
Garissa    171.0
Isiolo     281.0
Nairobi    714.0
Mombasa    768.0


## 5. county totals against the AFA national figures

In [21]:
county_totals = county_long.groupby("year")[["area_ha", "production_t"]].sum()
afa_national = national_afa.set_index("Year").loc[county_totals.index]

cross_check = pd.DataFrame({
    "county_sum_area_ha": county_totals["area_ha"],
    "afa_area_ha": afa_national["Area (ha)"],
    "county_sum_production_t": county_totals["production_t"],
    "afa_production_t": afa_national["Production (t)"],
})
cross_check["area_diff_pct"] = (cross_check["county_sum_area_ha"] / cross_check["afa_area_ha"] - 1) * 100
cross_check["production_diff_pct"] = (
    cross_check["county_sum_production_t"] / cross_check["afa_production_t"] - 1
) * 100
cross_check.round(2)

,county_sum_area_ha,afa_area_ha,county_sum_production_t,afa_production_t,area_diff_pct,production_diff_pct
2020,2171697,2171695.0,3795175,3795170,0.00,0.00
2021,2170503,2168603.0,3311941,3314430,0.09,-0.08
2022,2113520,2171980.0,3082220,3089901,-2.69,-0.25
2023,2430013,2552931.0,4285206,4285206,-4.81,0.00
2024,2407245,2414535.5,4028320,4028320,-0.30,0.00


## 6. Clean the national sheets

In [22]:
national_afa_clean = (
    national_afa.rename(
        columns={
            "Year": "year",
            "Area (ha)": "area_ha",
            "Production (t)": "production_t",
            "Yield (90-kg bags per ha)": "yield_bags_ha",
            "Farm-gate price (KSh per 90-kg bag)": "farmgate_price_ksh_bag",
            "Value of production (KSh billion)": "value_ksh_bn",
            "Imports (t)": "imports_t",
        },
        errors="raise",
    )
    .sort_values("year")
    .reset_index(drop=True)
)

# Check the reported yield against production / area (t/ha converted to 90-kg bags/ha)
national_afa_clean["yield_bags_ha_calc"] = national_afa_clean["production_t"] / 0.09 / national_afa_clean["area_ha"]
national_afa_clean[["year", "yield_bags_ha", "yield_bags_ha_calc"]].round(2)

,year,yield_bags_ha,yield_bags_ha_calc
0,2019,20.00,20.04
1,2020,19.00,19.42
2,2021,17.00,16.98
3,2022,16.00,15.81
4,2023,22.80,18.65
5,2024,18.53,18.54


In [23]:
national_es_clean = (
    national_es.rename(
        columns={
            "Year": "year",
            "Production (million 90-kg bags)": "production_m_bags",
            "Production ('000 t)": "production_kt",
            "Price to farmers (KSh per 100 kg)": "farmer_price_ksh_100kg",
            "Price to farmers (KSh per 90-kg bag)": "farmer_price_ksh_bag",
            "Retail price March (KSh/kg)": "retail_price_march_ksh_kg",
            "Retail price September (KSh/kg)": "retail_price_sept_ksh_kg",
            "Bought by NCPB and millers ('000 t)": "bought_ncpb_millers_kt",
        },
        errors="raise",
    )
    .sort_values("year")
    .reset_index(drop=True)
)
national_es_clean["provisional"] = national_es_clean["year"] == 2025

tonnes_ok = np.allclose(national_es_clean["production_kt"], national_es_clean["production_m_bags"] * 90, atol=1)
price_ok = np.allclose(
    national_es_clean["farmer_price_ksh_bag"], national_es_clean["farmer_price_ksh_100kg"] * 0.9, atol=1
)
print("Production ('000 t) = million bags x 90:", tonnes_ok)
print("Price per 90-kg bag = price per 100 kg x 0.9:", price_ok)

Production ('000 t) = million bags x 90: True
Price per 90-kg bag = price per 100 kg x 0.9: True


In [24]:
supply_use_clean = (
    supply_use.rename(
        columns={
            "Year": "year",
            "Production ('000 t)": "production_kt",
            "Imports ('000 t)": "imports_kt",
            "Exports ('000 t)": "exports_kt",
            "Stock change ('000 t)": "stock_change_kt",
            "Domestic supply ('000 t)": "domestic_supply_kt",
            "Processed ('000 t)": "processed_kt",
            "Losses ('000 t)": "losses_kt",
            "Feed ('000 t)": "feed_kt",
            "Seed ('000 t)": "seed_kt",
            "Other uses ('000 t)": "other_uses_kt",
            "Food ('000 t)": "food_kt",
            "Food (kg per person per year)": "food_kg_per_person",
            "Calories per person per day": "calories_per_person_day",
        },
        errors="raise",
    )
    .sort_values("year")
    .reset_index(drop=True)
)
supply_use_clean["provisional"] = supply_use_clean["year"] == 2025

supply_side = (
    supply_use_clean["production_kt"]
    + supply_use_clean["imports_kt"]
    - supply_use_clean["exports_kt"]
    - supply_use_clean["stock_change_kt"]
)
use_cols = ["processed_kt", "losses_kt", "feed_kt", "seed_kt", "other_uses_kt", "food_kt"]
use_side = supply_use_clean[use_cols].sum(axis=1)

pd.DataFrame({
    "year": supply_use_clean["year"],
    "domestic_supply_kt": supply_use_clean["domestic_supply_kt"],
    "production+imports-exports-stock_change": supply_side,
    "sum_of_uses": use_side,
})

,year,domestic_supply_kt,production+imports-exports-stock_change,sum_of_uses
0,2024,4008,4007,4008
1,2025,3548,3548,3548


## 7. Save the cleaned data

In [25]:
county_long.to_csv(PROCESSED_DIR / "county_maize_long.csv", index=False)
national_afa_clean.to_csv(PROCESSED_DIR / "national_afa_clean.csv", index=False)
national_es_clean.to_csv(PROCESSED_DIR / "national_es_clean.csv", index=False)
supply_use_clean.to_csv(PROCESSED_DIR / "supply_use_clean.csv", index=False)